# Test chat

In [3]:
from langgraph.checkpoint.memory import InMemorySaver

from graph import build_graph
from personas import PERSONAS

# In-memory checkpointer: this chat is separate from the Discord conversations in checkpoints.sqlite, and it resets whenever this cell is re-run.
app = build_graph(InMemorySaver())
config = {"configurable": {"thread_id": "notebook-test"}, "recursion_limit": 15}


async def chat(text):
    """Same as discord_bot.handle_turn, minus Discord: sends one message and prints each agent's reply as its node finishes."""
    turn_input = {"messages": [{"speaker": "user", "content": f"Tester: {text}"}]}
    async for update in app.astream(turn_input, config, stream_mode="updates"):
        for node, node_output in update.items():
            print(f"[{node} finished]")
            for msg in node_output.get("messages", []):
                print(f"{PERSONAS[msg['speaker']]['username']}: {msg['content']}\n")

In [4]:
await chat("Can anyone describe the roles of the team?")

[lead finished]
Liam: Liam leads the job and coordinates the work. Stark researches facts and audience angles, Claire writes and revises the copy, and Paul turns the chosen copy into a video script and renders the avatar video.



# Test video generation

In [2]:
import asyncio
import os

import httpx
from IPython.display import Video, display

from tools import heygen_start, heygen_status

SCRIPT = "Hi. My name is Rob Rooney, CEO of Hyperlayer. Welcome to Hyperlayer!"


async def wallet():
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get("https://api.heygen.com/v3/users/me", headers={"x-api-key": os.environ["HEYGEN_API_KEY"]})
    resp.raise_for_status()
    return resp.json()["data"].get("wallet")


print("wallet before:", await wallet())

video_id = await heygen_start(SCRIPT)
print("render started:", video_id)

while True:
    video = await heygen_status(video_id)
    print("status:", video["status"])
    if video["status"] == "completed":
        break
    if video["status"] == "failed":
        raise RuntimeError(video["failure_message"])
    await asyncio.sleep(10)

print("duration:", video.get("duration"), "seconds")
print("wallet after:", await wallet())
print(video["video_url"])
display(Video(video["video_url"], embed=False, width=640))

wallet before: {'auto_reload': {'enabled': False}, 'currency': 'usd', 'remaining_balance': 5.0}
render started: 5d160b34427c8efa0f73826850d40396
status: pending
status: processing
status: processing
status: processing
status: processing
status: processing
status: processing
status: processing
status: processing
status: processing
status: completed
duration: 5.51184 seconds
wallet after: {'auto_reload': {'enabled': False}, 'currency': 'usd', 'remaining_balance': 4.8}
https://files2.heygen.ai/aws_pacific/avatar_tmp/a697a5e25fa34732a69a23639bd5fea1/5d160b34427c8efa0f73826850d40396.mp4?download_id=fe5a312b-fdeb-41dd-a2dd-075506af2a2f&Expires=1791472058&Signature=bdLRd2dAc1di3lzjW2ZFCwIij7tF2itEy2~Sch4ZG3pkwOnptaR4Hu300WARh8LcX8H53A8oY9t17zQ-M5cHeJlaUD15UcHctlQazNcKjlob45W1vG1xfsmY1zHLqJK-7tmrT-2VnFYPuL2y8yEogMi4XktKLBofclGoG0K2VNd0GcLHsRkB-q75qRUZZNmo-Xk3Rw-qHsqbBkEsoeqiN33Aleqr8LUxkw3rKOQ-TfFEVYIxphL87Lpi-OfR-5n89LOlL~b8iDje0tXs2J3YoQIcPBLSkdvDq1H82HAt4irUh1LHLyE5VzKGQplXskl02Ji6Vgk3Az99X